In [2]:
import pandas as pd

def build_pure_cross_species_vocab(original_vocab_csv, mapping_file, save_path):
    # 1. 读取原 CellFM 词表 (假设 27855 行) [cite: 71]
    df_old = pd.read_csv(original_vocab_csv)
    
    # 2. 加载人类 Gene Symbol 到 Ensembl ID 的映射表
    # 映射表可从 Ensembl BioMart 获取 [cite: 69, 1072]
    mapping = pd.read_csv(mapping_file) 
    symbol_to_id = dict(zip(mapping['symbol'], mapping['ensembl_id']))
    
    # 3. 翻译：保证前 27855 个位置的对应关系不动
    new_vocab = []
    for sym in df_old['gene_name']:
        new_vocab.append(symbol_to_id.get(sym, sym)) # 优先用 ID，没有则保留原名占位
        
    old_size = len(new_vocab)
    
    # 4. 追加小鼠特有基因 (根据 GeneCompass 方法：不通用的保留物种特有的 ID) [cite: 70]
    # mouse_only_ids 需要您预先从数据中提取
    mouse_only_ids = ["ENSMUSG00000000001", "ENSMUSG00000000028"] # 示例
    final_vocab = new_vocab + mouse_only_ids
    
    pd.DataFrame({'gene_id': final_vocab}).to_csv(save_path, index=False)
    return old_size, len(final_vocab)

# 执行
# OLD_SIZE, NEW_SIZE = build_pure_cross_species_vocab('csv/expand_gene_info.csv', 'human_map.csv', 'cross_vocab.csv')

In [3]:
import torch
import torch.nn as nn
from model import Cell_FM

def adapt_cellfm_to_cross_species(model, old_size, new_size, embed_dim=768):
    # 1. 备份旧的 Embedding 层
    old_emb = model.gene_embedding
    
    # 2. 创建扩容后的新层
    new_emb = nn.Embedding(new_size, embed_dim)
    
    # 3. 拷贝知识：继承原有的 27855 个基因的预训练权重 [cite: 41, 76]
    with torch.no_grad():
        new_emb.weight[:old_size, :] = old_emb.weight.data
        # 对新基因进行随机初始化
        nn.init.normal_(new_emb.weight[old_size:, :], std=0.02)
        
    # 4. 替换原模型中的层
    model.gene_embedding = new_emb
    return model

# 使用示例
# model = Cell_FM(...) 
# model.load_state_dict(torch.load('cellfm_weights.pth'))
# model = adapt_cellfm_to_cross_species(model, 27855, 29000)

ModuleNotFoundError: No module named 'model'